# Single query helpers

Manual workflow helpers for `public/data/stars/*.json`.

Use by adding new code cells like:

```python
getStar("Sirius", rank=0)
getStarSystem("alpha Centauri", ["alpha Cen A", "alpha Cen B"], rank=0)
```

Missing required numeric values use `PLACEHOLDER_NUMBER = 0` so files remain JSON-valid but obvious to edit. Optional missing values use `None`.

In [1]:
from __future__ import annotations

import json
import math
import pickle
import re
import unicodedata
from pathlib import Path
from typing import Any

import numpy as np

try:
    from astroquery.simbad import Simbad
except Exception as exc:
    Simbad = None
    print(f"astroquery SIMBAD unavailable: {exc}")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name in {"scripts", "notebooks"} else Path.cwd()
DATA_DIR = PROJECT_ROOT / "public" / "data" / "stars"
CACHE_DIR = PROJECT_ROOT / "notebooks" / "cache"
CACHE_PATH = CACHE_DIR / "single_query_cache.pkl"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

PLACEHOLDER_STRING = ""
PLACEHOLDER_NUMBER = 0
PLACEHOLDER_COLOR = [255, 255, 255]

if CACHE_PATH.exists():
    with CACHE_PATH.open("rb") as f:
        CACHE = pickle.load(f)
else:
    CACHE = {"simbad": {}}
CACHE.setdefault("simbad", {})
CACHE.setdefault("next_uid", 1)

def save_cache():
    with CACHE_PATH.open("wb") as f:
        pickle.dump(CACHE, f)

def cache_keys() -> list[str]:
    """List cached SIMBAD query keys."""
    return sorted(CACHE.get("simbad", {}).keys())

def remove_cache_entry(*identifiers: str) -> list[str]:
    """Remove exact SIMBAD cache keys. Example: remove_cache_entry("Proxima Centauri")."""
    removed = []
    for identifier in identifiers:
        key = identifier.strip()
        if key in CACHE.get("simbad", {}):
            del CACHE["simbad"][key]
            removed.append(key)
    if removed:
        save_cache()
    print(f"removed {len(removed)} cache entries: {removed}")
    return removed

def remove_cache_matching(pattern: str) -> list[str]:
    """Remove cached SIMBAD queries whose key contains pattern, case-insensitive."""
    needle = pattern.lower()
    matches = [key for key in cache_keys() if needle in key.lower()]
    return remove_cache_entry(*matches)

def clear_simbad_cache(confirm: bool = False) -> None:
    """Clear all SIMBAD query cache. Requires confirm=True."""
    if not confirm:
        print("Pass confirm=True to clear SIMBAD cache")
        return
    CACHE["simbad"] = {}
    save_cache()
    print("cleared SIMBAD cache")

def set_next_uid(uid: int) -> None:
    """Manually set next numeric uid, useful after deleting generated JSON files."""
    CACHE["next_uid"] = int(uid)
    save_cache()
    print(f"next_uid={CACHE['next_uid']}")

GREEK = {
    "alf": ("α", "alpha"), "alpha": ("α", "alpha"),
    "bet": ("β", "beta"), "beta": ("β", "beta"),
    "gam": ("γ", "gamma"), "gamma": ("γ", "gamma"),
    "del": ("δ", "delta"), "delta": ("δ", "delta"),
    "eps": ("ε", "epsilon"), "epsilon": ("ε", "epsilon"),
    "zet": ("ζ", "zeta"), "zeta": ("ζ", "zeta"),
    "eta": ("η", "eta"), "the": ("θ", "theta"), "theta": ("θ", "theta"),
    "iot": ("ι", "iota"), "iota": ("ι", "iota"),
    "kap": ("κ", "kappa"), "kappa": ("κ", "kappa"),
    "lam": ("λ", "lambda"), "lambda": ("λ", "lambda"),
    "mu.": ("μ", "mu"), "nu.": ("ν", "nu"), "ksi": ("ξ", "xi"),
    "omi": ("ο", "omicron"), "omicron": ("ο", "omicron"),
    "pi.": ("π", "pi"), "rho": ("ρ", "rho"),
    "sig": ("σ", "sigma"), "sigma": ("σ", "sigma"),
    "tau": ("τ", "tau"), "ups": ("υ", "upsilon"), "upsilon": ("υ", "upsilon"),
    "phi": ("φ", "phi"), "chi": ("χ", "chi"), "psi": ("ψ", "psi"),
    "ome": ("ω", "omega"), "omega": ("ω", "omega"),
}


# not used
GREEK_ABBR = {
    "alf": "Alpha", "bet": "Beta", "gam": "Gamma", "del": "Delta", "eps": "Epsilon", "zet": "Zeta", "eta": "Eta", "the": "Theta",
    "iot": "Iota", "kap": "Kappa", "lam": "Lambda", "mu.": "Mu", "nu.": "Nu", "ksi": "Xi", "omi": "Omicron", "pi.": "Pi",
    "rho": "Rho", "sig": "Sigma", "tau": "Tau", "ups": "Upsilon", "phi": "Phi", "khi": "Chi", "psi": "Psi", "ome": "Omega",
}
# not used
GREEK_ABBR_TO_UTF8 = {
    "alf": "α", "bet": "β", "gam": "γ", "del": "δ", "eps": "ε", "zet": "ζ", "eta": "η", "the": "θ",
    "iot": "ι", "kap": "κ", "lam": "λ", "mu.": "μ", "nu.": "ν", "ksi": "ξ", "omi": "ο", "pi.": "π",
    "rho": "ρ", "sig": "σ", "tau": "τ", "ups": "υ", "phi": "φ", "khi": "χ", "psi": "ψ", "ome": "ω",
}

CONSTELLATION_ABBR = {
    "And": "Andromedae", "Ant": "Antliae", "Aps": "Apodis", "Aql": "Aquilae", "Aqr": "Aquarii", "Ara": "Arae", "Ari": "Arietis", "Aur": "Aurigae",
    "Boo": "Bootis", "CMa": "Canis Majoris", "CMi": "Canis Minoris", "CVn": "Canum Venaticorum", "Cae": "Caeli", "Cam": "Camelopardalis",
    "Cap": "Capricorni", "Car": "Carinae", "Cas": "Cassiopeiae", "Cen": "Centauri", "Cep": "Cephei", "Cet": "Ceti", "Cha": "Chamaeleontis",
    "Cir": "Circini", "Cnc": "Cancri", "Col": "Columbae", "Com": "Comae Berenices", "CrA": "Coronae Australis", "CrB": "Coronae Borealis",
    "Crt": "Crateris", "Cru": "Crucis", "Crv": "Corvi", "Cyg": "Cygni", "Del": "Delphini", "Dor": "Doradus", "Dra": "Draconis",
    "Equ": "Equulei", "Eri": "Eridani", "For": "Fornacis", "Gem": "Geminorum", "Gru": "Gruis", "Her": "Herculis", "Hor": "Horologii",
    "Hya": "Hydrae", "Hyi": "Hydri", "Ind": "Indi", "LMi": "Leonis Minoris", "Lac": "Lacertae", "Leo": "Leonis", "Lep": "Leporis",
    "Lib": "Librae", "Lup": "Lupi", "Lyn": "Lyncis", "Lyr": "Lyrae", "Men": "Mensae", "Mic": "Microscopii", "Mon": "Monocerotis",
    "Mus": "Muscae", "Nor": "Normae", "Oct": "Octantis", "Oph": "Ophiuchi", "Ori": "Orionis", "Pav": "Pavonis", "Peg": "Pegasi",
    "Per": "Persei", "Phe": "Phoenicis", "Pic": "Pictoris", "PsA": "Piscis Austrini", "Psc": "Piscium", "Pup": "Puppis", "Pyx": "Pyxidis",
    "Ret": "Reticuli", "Scl": "Sculptoris", "Sco": "Scorpii", "Sct": "Scuti", "Ser": "Serpentis", "Sex": "Sextantis", "Sge": "Sagittae",
    "Sgr": "Sagittarii", "Tau": "Tauri", "Tel": "Telescopii", "TrA": "Trianguli Australis", "Tri": "Trianguli", "Tuc": "Tucanae",
    "UMa": "Ursae Majoris", "UMi": "Ursae Minoris", "Vel": "Velorum", "Vir": "Virginis", "Vol": "Volantis", "Vul": "Vulpeculae",
}
CONSTELLATION_ORDER = sorted(CONSTELLATION_ABBR)
CONSTELLATION_INDEX = {abbr: i for i, abbr in enumerate(CONSTELLATION_ORDER)}



In [2]:
def clean_value(v: Any) -> Any:
    if v is None:
        return None
    if isinstance(v, np.ma.MaskedArray) or getattr(v, "mask", False) is np.ma.masked:
        return None
    if hasattr(v, "item"):
        try:
            v = v.item()
        except Exception:
            pass
    if isinstance(v, bytes):
        v = v.decode("utf-8", errors="replace")
    if isinstance(v, float) and (math.isnan(v) or math.isinf(v)):
        return None
    return v

def table_value(row: Any, *names: str) -> Any:
    if row is None:
        return None
    colnames = list(getattr(row, "colnames", [])) or list(getattr(row, "keys", lambda: [])())
    lower = {c.lower(): c for c in colnames}
    for name in names:
        key = lower.get(name.lower())
        if key is None:
            continue
        value = clean_value(row[key])
        if value not in (None, ""):
            return value
    return None

def slugify(text: str) -> str:
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = text.lower().replace("+", " plus ")
    text = re.sub(r"[^a-z0-9]+", "_", text).strip("_")
    return re.sub(r"_+", "_", text) or "unnamed"

def unique_path(filename_base: str) -> tuple[str, Path]:
    base = slugify(filename_base)
    path = DATA_DIR / f"{base}.json"
    if not path.exists():
        return base, path
    i = 2
    while True:
        candidate = DATA_DIR / f"{base}_{i}.json"
        if not candidate.exists():
            return f"{base}_{i}", candidate
        i += 1

def relevant_simbad_ids(ids_blob: str | None, main_id: str | None = None) -> list[str]:
    ids = []
    if main_id:
        ids.append(str(main_id).strip())
    if ids_blob:
        ids.extend(x.strip() for x in str(ids_blob).split("|") if x.strip())
    # Keep order, drop duplicates.
    seen = set()
    out = []
    for ident in ids:
        key = ident.upper()
        if key not in seen:
            out.append(ident)
            seen.add(key)
    return out

def next_uid() -> int:
    uid = int(CACHE.get("next_uid", 1))
    CACHE["next_uid"] = uid + 1
    save_cache()
    return uid

def constellation_from_text(text: str | None) -> str | None:
    if not text:
        return None
    text = normalize_designation(text) or ""
    # Prefer exact 3-letter IAU abbrev tokens.
    for abbr in CONSTELLATION_ORDER:
        if re.search(rf"(?<![A-Za-z]){re.escape(abbr)}(?![A-Za-z])", text):
            return abbr
    # Fallback: genitive/full names.
    for abbr, name in CONSTELLATION_ABBR.items():
        if name.lower() in text.lower():
            return abbr
    return None

def constellation_index_from_ids(ids_blob: str | None, main_id: str | None = None) -> int:
    joined = " | ".join(relevant_simbad_ids(ids_blob, main_id))
    abbr = constellation_from_text(joined)
    return CONSTELLATION_INDEX.get(abbr, -1)

def parse_ids(ids_blob: str | None, main_id: str | None = None) -> dict[str, Any]:
    """Keep only catalog ids we use or may query later. No all-ids array."""
    out = {
        "simbad_main_id": main_id,
        "hip_id": None,
        "hd_id": None,
        "gaia_dr2_id": None,
        "gaia_dr3_id": None,
        "gj_id": None,
        "tyc_id": None,
        "2mass_id": None,
    }
    for ident in relevant_simbad_ids(ids_blob, main_id):
        u = ident.upper().replace("  ", " ")
        if u.startswith("HIP ") and out["hip_id"] is None:
            out["hip_id"] = ident
        elif u.startswith("HD ") and out["hd_id"] is None:
            out["hd_id"] = ident
        elif u.startswith("GAIA DR2") and out["gaia_dr2_id"] is None:
            out["gaia_dr2_id"] = ident
        elif u.startswith("GAIA DR3") and out["gaia_dr3_id"] is None:
            out["gaia_dr3_id"] = ident
        elif (u.startswith("GJ ") or u.startswith("GL ") or u.startswith("GLIESE")) and out["gj_id"] is None:
            out["gj_id"] = ident
        elif u.startswith("TYC ") and out["tyc_id"] is None:
            out["tyc_id"] = ident
        elif u.startswith("2MASS") and out["2mass_id"] is None:
            out["2mass_id"] = ident
    return out

def proper_name_from_ids(ids_blob: str | None) -> str | None:
    if not ids_blob:
        return None
    names = []
    for ident in [x.strip() for x in str(ids_blob).split("|")]:
        if ident.upper().startswith("NAME "):
            name = ident[5:].strip()
            if name and not re.search(r"^(NAME|Cl\*|NAME LMC|NAME SMC)", name, re.I):
                names.append(name)
    return max(names, key=len) if names else None

SUPERSCRIPT_TO_ASCII = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻", "0123456789+-")
ASCII_TO_SUPERSCRIPT = str.maketrans("0123456789+-", "⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻")

def superscript_to_ascii(text: str | None) -> str | None:
    """Convert α¹ Cen -> α1 Cen, useful before parsing."""
    return None if text is None else str(text).translate(SUPERSCRIPT_TO_ASCII)

def ascii_to_superscript(text: str | int | None) -> str | None:
    """Convert 12 -> ¹² for Bayer suffix display."""
    return None if text is None else str(text).translate(ASCII_TO_SUPERSCRIPT)

def strip_star_prefix(ident: str | None) -> str | None:
    """SIMBAD star ids often start with '* ', e.g. '* alf CMa'."""
    if ident is None:
        return None
    return re.sub(r"^\s*\*\s*", "", str(ident).strip())

def normalize_designation(ident: str | None) -> str | None:
    """Remove SIMBAD star prefix and normalize unicode superscripts to ASCII digits."""
    if ident is None:
        return None
    return superscript_to_ascii(strip_star_prefix(ident))

def greek_to_latin_text(text: str | None) -> str | None:
    """Convert Greek Bayer symbols anywhere in text to latin names for parsing/search."""
    if text is None:
        return None
    out = str(text)
    for _key, (symbol, latin) in GREEK.items():
        out = out.replace(symbol, latin)
    return out

def split_bayer_abbr(abbr: str | None) -> tuple[str, str, str, str] | None:
    """Return (greek_key, numeric_suffix, constellation_abbr, component_letter) from forms like 'alf CMa', '* alf01 Cen A', 'α¹ Cen'."""
    if not abbr:
        return None
    text = greek_to_latin_text(normalize_designation(abbr))
    m = re.match(r"^([A-Za-z]{2,7})\s*([0-9]*)\s+([A-Z][a-zA-Z]{2})(?:\s+([A-Z]))?$", text)
    if not m:
        return None
    greek_key, suffix, con, component = m.groups()
    suffix = suffix.lstrip("0") or ("0" if suffix else "")
    if greek_key.lower() not in GREEK:
        return None
    return greek_key.lower(), suffix, con, component or ""

def find_bayer_candidates(text: str | None) -> list[dict[str, str]]:
    """Find Bayer-like substrings in combined SIMBAD ids. Component-letter matches rank above numeric superscript matches."""
    if not text:
        return []
    text = greek_to_latin_text(superscript_to_ascii(strip_star_prefix(text)))
    pattern = re.compile(r"(?<![A-Za-z0-9])([A-Za-z]{2,7})\s*([0-9]*)\s+([A-Z][a-zA-Z]{2})(?:\s+([A-Z]))?(?![A-Za-z0-9])")
    candidates = []
    for m in pattern.finditer(text):
        greek_key, suffix, con, component = m.groups()
        if greek_key.lower() not in GREEK:
            continue
        suffix = suffix.lstrip("0") or ("0" if suffix else "")
        component = component or ""
        candidates.append({
            "abbr": f"{greek_key} {con}{(' ' + component) if component else ''}" if not suffix else f"{greek_key}{suffix} {con}{(' ' + component) if component else ''}",
            "greek_key": greek_key.lower(),
            "suffix": suffix,
            "con": con,
            "component": component,
        })
    candidates.sort(key=lambda c: (bool(c["component"]), not bool(c["suffix"])), reverse=True)
    return candidates

def best_bayer_candidate(ids_blob: str | None, main_id: str | None = None) -> dict[str, str] | None:
    joined = " | ".join(relevant_simbad_ids(ids_blob, main_id))
    candidates = find_bayer_candidates(joined)
    return candidates[0] if candidates else None

def bayer_candidate_to_greek(candidate: dict[str, str], superscript: bool = True) -> str:
    symbol, _latin = GREEK[candidate["greek_key"]]
    suffix_text = ascii_to_superscript(candidate["suffix"]) if superscript else candidate["suffix"]
    component = f" {candidate['component']}" if candidate.get("component") else ""
    return f"{symbol}{suffix_text or ''} {CONSTELLATION_ABBR.get(candidate['con'], candidate['con'])}{component}"

def bayer_candidate_to_latin(candidate: dict[str, str]) -> str:
    _symbol, latin = GREEK[candidate["greek_key"]]
    component = f" {candidate['component']}" if candidate.get("component") else ""
    return f"{latin}{candidate['suffix'] or ''} {CONSTELLATION_ABBR.get(candidate['con'], candidate['con'])}{component}"

def bayer_abbr_to_greek(abbr: str | None, superscript: bool = True) -> str | None:
    """'alf 1 Cen A' or '* alf01 Cen' -> 'α Centauri A' / 'α¹ Centauri'."""
    parts = split_bayer_abbr(abbr)
    if not parts:
        return None
    greek_key, suffix, con, component = parts
    return bayer_candidate_to_greek({"greek_key": greek_key, "suffix": suffix, "con": con, "component": component}, superscript=superscript)

def bayer_abbr_to_latin(abbr: str | None) -> str | None:
    """'alf 1 Cen A' or '* alf01 Cen' -> 'alpha Centauri A' / 'alpha1 Centauri'. Filename-friendly Bayer."""
    parts = split_bayer_abbr(abbr)
    if not parts:
        return None
    greek_key, suffix, con, component = parts
    return bayer_candidate_to_latin({"greek_key": greek_key, "suffix": suffix, "con": con, "component": component})

def make_bayer_abbr(greek: str, constellation_abbr: str, suffix: str | int | None = None, component: str | None = None) -> str:
    """make_bayer_abbr('alpha', 'Cen', component='A') -> 'alf Cen A'."""
    greek_l = greek.lower()
    # Prefer common SIMBAD 3-letter keys where available.
    reverse = {latin: key for key, (_symbol, latin) in GREEK.items() if len(key) == 3}
    key = reverse.get(greek_l, greek_l)
    return f"{key}{'' if suffix is None else suffix} {constellation_abbr}{'' if component is None else ' ' + component}"

def split_flamsteed_abbr(abbr: str | None) -> tuple[str, str] | None:
    text = normalize_designation(abbr)
    if not text:
        return None
    m = re.match(r"^(\d+)\s+([A-Z][a-zA-Z]{2})$", text)
    return m.groups() if m else None

def flamsteed_abbr_to_name(abbr: str | None) -> str | None:
    parts = split_flamsteed_abbr(abbr)
    if not parts:
        return None
    n, con = parts
    return f"{n} {CONSTELLATION_ABBR.get(con, con)}"

def make_flamsteed_abbr(number: int | str, constellation_abbr: str) -> str:
    return f"{number} {constellation_abbr}"

def find_bayer_flamsteed(ids_blob: str | None, main_id: str | None = None) -> tuple[str | None, str | None, str | None, str | None, str | None]:
    bayer_abbr = flam_abbr = flam_name = bayer_greek = bayer_name = None
    unresolved_star_id = None
    ids = relevant_simbad_ids(ids_blob, main_id)
    for ident in ids:
        if str(ident).lstrip().startswith("*") and unresolved_star_id is None:
            unresolved_star_id = ident

    # Search across all candidate strings at once so '* alf Cen A' wins over 'alf01 Cen'.
    # Priority: component-letter Bayer > plain Bayer > numeric/superscript Bayer.
    candidate = best_bayer_candidate(ids_blob, main_id)
    if candidate:
        bayer_abbr = candidate["abbr"]
        bayer_greek = bayer_candidate_to_greek(candidate)
        bayer_name = bayer_candidate_to_latin(candidate)

    for ident in ids:
        if flam_abbr is None and split_flamsteed_abbr(ident):
            flam_abbr = normalize_designation(ident)
            flam_name = flamsteed_abbr_to_name(ident)

    # If SIMBAD main id was '* ...' but parser failed, keep it in Bayer fields for manual cleanup.
    if bayer_abbr is None and flam_abbr is None and unresolved_star_id is not None:
        bayer_abbr = unresolved_star_id
        bayer_greek = unresolved_star_id
        bayer_name = unresolved_star_id
    return bayer_abbr, bayer_name, bayer_greek, flam_abbr, flam_name

def choose_filename_name(names: dict[str, Any]) -> str:
    if names.get("proper_name"):
        return names["proper_name"]
    if names.get("bayer_name"):
        return names["bayer_name"]
    if names.get("bayer_greek"):
        return bayer_abbr_to_latin(names.get("bayer_abbr")) or names["bayer_greek"]
    if names.get("flamsteed_name"):
        return names["flamsteed_name"]
    return "unnamed"


In [3]:
def parallax_to_distance_pc(parallax_mas: float | None) -> float | None:
    if parallax_mas is None or parallax_mas <= 0:
        return None
    return 1000.0 / parallax_mas

def distance_pc_to_parallax_mas(distance_pc: float | None) -> float | None:
    if distance_pc is None or distance_pc <= 0:
        return None
    return 1000.0 / distance_pc

def apparent_to_absolute_mag(apparent_mag: float | None, distance_pc: float | None) -> float | None:
    if apparent_mag is None or distance_pc is None or distance_pc <= 0:
        return None
    return apparent_mag - 5.0 * math.log10(distance_pc / 10.0)

def bv_to_teff_k(bv: float | None) -> float | None:
    # Ballesteros 2012 approximation. Good enough for color placeholder.
    if bv is None:
        return None
    return 4600.0 * ((1.0 / (0.92 * bv + 1.7)) + (1.0 / (0.92 * bv + 0.62)))

def teff_to_rgb(teff_k: float | None) -> list[int] | None:
    if teff_k is None or teff_k <= 0:
        return None
    t = max(1000.0, min(40000.0, float(teff_k))) / 100.0
    if t <= 66.0:
        r = 255.0
        g = 99.4708025861 * math.log(t) - 161.1195681661
        b = 0.0 if t <= 19.0 else 138.5177312231 * math.log(t - 10.0) - 305.0447927307
    else:
        r = 329.698727446 * ((t - 60.0) ** -0.1332047592)
        g = 288.1221695283 * ((t - 60.0) ** -0.0755148492)
        b = 255.0
    return [int(round(max(0, min(255, x)))) for x in (r, g, b)]

def bv_to_rgb(bv: float | None) -> list[int] | None:
    return teff_to_rgb(bv_to_teff_k(bv))

def spectral_type_to_teff_k(sp: str | None) -> float | None:
    if not sp:
        return None
    # rough mid-class estimates, useful only when SIMBAD lacks B/V and teff.
    letter = sp.strip().upper()[:1]
    return {"O": 30000, "B": 15000, "A": 8500, "F": 6500, "G": 5500, "K": 4300, "M": 3200, "L": 2100, "T": 1200, "Y": 500}.get(letter)

def infer_evolution_stage(spectral_type: str | None, otype: str | None = None) -> str:
    """Broad stage only: dwarf/subgiant/giant/supergiant/remnant/unknown."""
    text = f"{spectral_type or ''} {otype or ''}".upper()
    if "WD" in text or "WHITE DWARF" in text:
        return "remnant"
    if re.search(r"\b(IA|IAB|IB|I|II)\b", text):
        return "supergiant"
    if "III" in text:
        return "giant"
    if "IV" in text:
        return "subgiant"
    if "V" in text:
        return "dwarf"
    return PLACEHOLDER_STRING

def infer_star_type(spectral_type: str | None, otype: str | None = None) -> str:
    """More specific display class: red giant, white dwarf, red dwarf, blue supergiant, etc."""
    text = f"{spectral_type or ''} {otype or ''}".upper()
    letter = (spectral_type or "").strip().upper()[:1]
    color = {
        "O": "blue", "B": "blue-white", "A": "white", "F": "yellow-white",
        "G": "yellow", "K": "orange", "M": "red", "L": "brown", "T": "brown", "Y": "brown",
    }.get(letter, "")

    if "WD" in text or "WHITE DWARF" in text:
        return "white dwarf"
    if re.search(r"\b(IA|IAB|IB|I|II)\b", text):
        return f"{color} supergiant".strip()
    if "III" in text:
        return f"{color} giant".strip()
    if "IV" in text:
        return f"{color} subgiant".strip()
    if "V" in text:
        return f"{color} dwarf".strip()
    if letter in {"L", "T", "Y"}:
        return "brown dwarf"
    return PLACEHOLDER_STRING


In [4]:
def make_simbad_client() -> Any:
    if Simbad is None:
        raise RuntimeError("astroquery.simbad not available")
    simbad = Simbad()
    # Fields vary by astroquery/SIMBAD version; ignore unsupported names.
    for field in ["ids", "otype", "sp_type", "parallax", "flux(B)", "flux(V)", "flux(G)", "mesdistance"]:
        try:
            simbad.add_votable_fields(field)
        except Exception:
            pass
    return simbad

def query_simbad(identifier: str, refresh: bool = False) -> dict[str, Any]:
    key = identifier.strip()
    if not refresh and key in CACHE["simbad"]:
        return CACHE["simbad"][key]
    simbad = make_simbad_client()
    table = simbad.query_object(identifier)
    if table is None or len(table) == 0:
        data = {"query_id": identifier, "found": False}
    else:
        row = table[0]
        data = {
            "query_id": identifier,
            "found": True,
            "main_id": table_value(row, "MAIN_ID", "main_id"),
            "ra": table_value(row, "RA", "ra"),
            "dec": table_value(row, "DEC", "dec"),
            "ra_deg": table_value(row, "RA_d", "ra_d"),
            "dec_deg": table_value(row, "DEC_d", "dec_d"),
            "otype": table_value(row, "OTYPE", "otype"),
            "spectral_type": table_value(row, "SP_TYPE", "sp_type", "SP_TYPE_2"),
            "parallax_mas": table_value(row, "PLX_VALUE", "plx_value", "PLX"),
            "flux_b": table_value(row, "FLUX_B", "flux_b"),
            "flux_v": table_value(row, "FLUX_V", "flux_v"),
            "flux_g": table_value(row, "FLUX_G", "flux_g"),
            "ids_blob": table_value(row, "IDS", "ids"),
        }
    CACHE["simbad"][key] = data
    save_cache()
    return data

def entry_names_from_simbad(data: dict[str, Any]) -> dict[str, Any]:
    ids_blob = data.get("ids_blob")
    bayer_abbr, bayer_name, bayer_greek, flam_abbr, flam_name = find_bayer_flamsteed(ids_blob, data.get("main_id"))
    return {
        "proper_name": proper_name_from_ids(ids_blob),
        "bayer_abbr": bayer_abbr,
        "bayer_name": bayer_name,
        "bayer_greek": bayer_greek,
        "flamsteed_abbr": flam_abbr,
        "flamsteed_name": flam_name,
    }

def compact_ids(ids: dict[str, Any]) -> dict[str, str]:
    return {k: str(v).strip().upper() for k, v in ids.items() if v not in (None, "")}

def ids_overlap(a: dict[str, Any], b: dict[str, Any]) -> set[str]:
    """Return matching catalog id field names."""
    aa = compact_ids(a)
    bb = compact_ids(b)
    return {k for k, v in aa.items() if k in bb and bb[k] == v}

def simbad_aliases(data: dict[str, Any]) -> set[str]:
    """All SIMBAD identifiers from current query/cache, for duplicate detection only. Not written to JSON."""
    return {x.strip().upper() for x in relevant_simbad_ids(data.get("ids_blob"), data.get("main_id")) if x and x.strip()}

def entry_aliases(entry: dict[str, Any]) -> set[str]:
    """Identifiers implied by an existing JSON entry, for duplicate detection only."""
    aliases = set(compact_ids(entry.get("ids", {})).values())
    names = entry.get("names", {}) or {}
    if names.get("proper_name"):
        aliases.add(f"NAME {names['proper_name']}".upper())
    if names.get("bayer_abbr"):
        aliases.add(str(names["bayer_abbr"]).upper())
        aliases.add(f"* {names['bayer_abbr']}".upper())
    if names.get("flamsteed_abbr"):
        aliases.add(str(names["flamsteed_abbr"]).upper())
        aliases.add(f"* {names['flamsteed_abbr']}".upper())
    return aliases

def find_existing_star_by_ids(ids: dict[str, Any]) -> tuple[Path, dict[str, Any]] | tuple[None, None]:
    """Scan generated JSON files for same catalog id to avoid duplicate stars."""
    if not compact_ids(ids):
        return None, None
    for path in sorted(DATA_DIR.glob("*.json")):
        try:
            entry = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            continue
        if entry.get("object_type") != "single_star":
            continue
        if ids_overlap(ids, entry.get("ids", {})):
            return path, entry
    return None, None

def find_existing_star_by_simbad_data(data: dict[str, Any], ids: dict[str, Any] | None = None) -> tuple[Path, dict[str, Any]] | tuple[None, None]:
    """Find existing star using catalog ids plus SIMBAD aliases from current query/cache."""
    ids = ids or parse_ids(data.get("ids_blob"), data.get("main_id"))
    path, entry = find_existing_star_by_ids(ids)
    if entry is not None:
        return path, entry
    aliases = simbad_aliases(data)
    if not aliases:
        return None, None
    for path in sorted(DATA_DIR.glob("*.json")):
        try:
            entry = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            continue
        if entry.get("object_type") != "single_star":
            continue
        if aliases & entry_aliases(entry):
            return path, entry
    return None, None

def reuse_existing_star_if_found(data: dict[str, Any], ids: dict[str, Any] | None = None, parent_uid: int | None = None) -> dict[str, Any] | None:
    """Return existing entry instead of duplicating.

    If parent_uid is provided, mark the existing star as a child of that parent.
    Only is_child and parent_uid are changed; all other fields are preserved.
    """
    path, entry = find_existing_star_by_simbad_data(data, ids)
    if entry is None:
        return None
    if parent_uid is not None:
        changed = entry.get("is_child") is not True or entry.get("parent_uid") != parent_uid
        entry["is_child"] = True
        entry["parent_uid"] = parent_uid
        if changed:
            path.write_text(json.dumps(entry, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")
            print(f"updated parent uid={entry.get('uid')} {path.relative_to(PROJECT_ROOT)}")
    print(f"reused existing uid={entry.get('uid')} {path.relative_to(PROJECT_ROOT)}")
    return entry

def find_entry_by_uid(uid: int) -> tuple[Path, dict[str, Any]] | tuple[None, None]:
    """Find generated JSON entry by numeric uid."""
    uid = int(uid)
    for path in sorted(DATA_DIR.glob("*.json")):
        try:
            entry = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            continue
        if entry.get("uid") == uid:
            return path, entry
    return None, None

def remove_entry_by_uid(uid: int, confirm: bool = False, unlink_from_systems: bool = True) -> Path | None:
    """Delete generated JSON file by uid. Requires confirm=True.

    If unlink_from_systems=True, removes uid from any system child_uids and clears parent links on children when deleting a system.
    Cache is not changed except next_uid remains monotonic.
    """
    if not confirm:
        print(f"Pass confirm=True to delete uid={uid}")
        return None
    uid = int(uid)
    path, entry = find_entry_by_uid(uid)
    if entry is None:
        print(f"uid={uid} not found")
        return None

    if unlink_from_systems:
        # Remove deleted object from any system child list.
        for other_path in sorted(DATA_DIR.glob("*.json")):
            if other_path == path:
                continue
            try:
                other = json.loads(other_path.read_text(encoding="utf-8"))
            except Exception:
                continue
            changed = False
            if other.get("object_type") == "star_system" and uid in other.get("child_uids", []):
                other["child_uids"] = [x for x in other.get("child_uids", []) if x != uid]
                changed = True
            # If deleting a system, detach its children instead of deleting them.
            if entry.get("object_type") == "star_system":
                if other.get("is_child") and other.get("parent_uid") == uid:
                    other["is_child"] = False
                    other.pop("parent_uid", None)
                    other.setdefault("rank", 4)
                    changed = True
            if changed:
                other_path.write_text(json.dumps(other, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")
                print(f"updated {other_path.relative_to(PROJECT_ROOT)}")

    path.unlink()
    print(f"deleted uid={uid} {path.relative_to(PROJECT_ROOT)}")
    return path

def write_json_entry(entry: dict[str, Any], overwrite: bool = False, filename_base: str | None = None) -> Path:
    if "uid" not in entry:
        entry["uid"] = next_uid()
    filename_base = filename_base or choose_filename_name(entry["names"]) or f"uid_{entry['uid']}"
    if overwrite:
        filename_base = slugify(filename_base)
        path = DATA_DIR / f"{filename_base}.json"
    else:
        _slug, path = unique_path(filename_base)
    path.write_text(json.dumps(entry, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")
    print(f"wrote uid={entry['uid']} {path.relative_to(PROJECT_ROOT)}")
    return path


In [5]:
def _first_value_with_source(child_data: dict[str, Any], fallback_data: dict[str, Any] | None, key: str, child_source: str = "SIMBAD", fallback_source: str = "SIMBAD:parent_system") -> tuple[Any, str | None]:
    value = clean_value(child_data.get(key))
    if value is not None:
        return value, child_source
    if fallback_data:
        value = clean_value(fallback_data.get(key))
        if value is not None:
            return value, fallback_source
    return None, None

def _bv_with_source(child_data: dict[str, Any], fallback_data: dict[str, Any] | None) -> tuple[float | None, str | None, Any, str | None]:
    b = clean_value(child_data.get("flux_b"))
    v = clean_value(child_data.get("flux_v"))
    if b is not None and v is not None:
        return float(b) - float(v), "SIMBAD:flux(B)-flux(V)", v, "SIMBAD:flux(V)"
    if fallback_data:
        b = clean_value(fallback_data.get("flux_b"))
        v = clean_value(fallback_data.get("flux_v"))
        if b is not None and v is not None:
            return float(b) - float(v), "SIMBAD:parent_system:flux(B)-flux(V)", v, "SIMBAD:parent_system:flux(V)"
    return None, None, v, "SIMBAD:flux(V)" if v is not None else None

def getStar(identifier: str, parent_uid: int | None = None, rank: int = 4, fallback_data: dict[str, Any] | None = None, refresh: bool = False, overwrite: bool = False) -> dict[str, Any]:
    data = query_simbad(identifier, refresh=refresh)
    fallback_data = fallback_data or {}
    names = entry_names_from_simbad(data)
    ids = parse_ids(data.get("ids_blob"), data.get("main_id"))
    existing = reuse_existing_star_if_found(data, ids, parent_uid=parent_uid)
    if existing is not None:
        return existing

    ra_deg, ra_source = _first_value_with_source(data, fallback_data, "ra_deg")
    dec_deg, dec_source = _first_value_with_source(data, fallback_data, "dec_deg")
    parallax, parallax_source = _first_value_with_source(data, fallback_data, "parallax_mas")
    distance = parallax_to_distance_pc(parallax)
    distance_source = "derived:parallax_mas" if distance is not None else None

    bv, bv_source, v_mag, v_source = _bv_with_source(data, fallback_data)
    spectral_type = clean_value(data.get("spectral_type"))
    teff = bv_to_teff_k(bv) or spectral_type_to_teff_k(spectral_type)
    teff_source = "derived:bv_color_index" if bv is not None else ("derived:spectral_type" if teff else None)
    rgb = bv_to_rgb(bv) or teff_to_rgb(teff)
    rgb_source = "derived:bv_color_index" if bv is not None else ("derived:spectral_type" if rgb else None)
    absmag = apparent_to_absolute_mag(float(v_mag), distance) if v_mag is not None and distance is not None else None
    absmag_source = "derived:apparent_mag,distance_pc" if absmag is not None else None

    # Parent-system fallback only applies to numeric/derived numeric fields.
    # Names, ids, spectral_type, and parent fields still describe this child star.
    entry = {
        "object_type": "single_star",
        "uid": next_uid(),
        "constellation": constellation_index_from_ids(data.get("ids_blob"), data.get("main_id")),
        "names": names,
        "ids": ids,
        "is_child": parent_uid is not None,
        "ra_deg": float(ra_deg) if ra_deg is not None else PLACEHOLDER_NUMBER,
        "ra_deg_source": ra_source or PLACEHOLDER_STRING,
        "dec_deg": float(dec_deg) if dec_deg is not None else PLACEHOLDER_NUMBER,
        "dec_deg_source": dec_source or PLACEHOLDER_STRING,
        "distance_pc": float(distance) if distance is not None else PLACEHOLDER_NUMBER,
        "distance_pc_source": distance_source or PLACEHOLDER_STRING,
        "parallax_mas": float(parallax) if parallax is not None else None,
        "parallax_mas_source": parallax_source,
        "radius_sol": None, "radius_sol_source": None,
        "mass_sol": None, "mass_sol_source": None,
        "color_rgb": rgb or PLACEHOLDER_COLOR,
        "color_rgb_source": rgb_source or PLACEHOLDER_STRING,
        "teff_k": float(teff) if teff is not None else None,
        "teff_k_source": teff_source,
        "bv_color_index": float(bv) if bv is not None else None,
        "bv_color_index_source": bv_source,
        "absmag": absmag,
        "absmag_source": absmag_source,
        "luminosity_sol": None, "luminosity_sol_source": None,
        "spectral_type": str(spectral_type) if spectral_type else PLACEHOLDER_STRING,
        "spectral_type_source": "SIMBAD" if spectral_type else PLACEHOLDER_STRING,
        "evolution_stage": infer_evolution_stage(spectral_type, data.get("otype")),
        "evolution_stage_source": "derived:spectral_type/otype",
        "star_type": infer_star_type(spectral_type, data.get("otype")),
        "star_type_source": "derived:spectral_type/otype",
        "metallicity_dex": None, "metallicity_dex_source": None,
        "age_yr": None, "age_yr_source": None,
        "remarks": None,
    }
    if parent_uid is not None:
        entry["parent_uid"] = parent_uid
    entry["rank"] = rank
    write_json_entry(entry, overwrite=overwrite, filename_base=choose_filename_name(names) if choose_filename_name(names) != "unnamed" else identifier)
    return entry

def getStarSystem(identifier: str, children: list[str], parent_uid: int | None = None, rank: int = 4, refresh: bool = False, overwrite: bool = False) -> dict[str, Any]:
    data = query_simbad(identifier, refresh=refresh)
    names = entry_names_from_simbad(data)
    ids = parse_ids(data.get("ids_blob"), data.get("main_id"))
    system_uid = next_uid()

    child_entries = []
    for i, child in enumerate(children):
        # Primary star gets parent system numeric data as fallback.
        # Secondary/other children do not, because system astrometry/photometry usually describes photocenter or combined light.
        child_fallback = data if i == 0 else None
        child_entry = getStar(child, parent_uid=system_uid, rank=4, fallback_data=child_fallback, refresh=refresh, overwrite=overwrite)
        child_entries.append(child_entry)

    entry = {
        "object_type": "star_system",
        "uid": system_uid,
        "constellation": constellation_index_from_ids(data.get("ids_blob"), data.get("main_id")),
        "names": names,
        "ids": ids,
        "is_child": parent_uid is not None,
        "child_uids": [c["uid"] for c in child_entries],
        "remarks": None,
    }
    if parent_uid is not None:
        entry["parent_uid"] = parent_uid
    entry["rank"] = rank
    write_json_entry(entry, overwrite=overwrite, filename_base=choose_filename_name(names) if choose_filename_name(names) != "unnamed" else identifier)
    return entry


## Useful optional fields to consider later

For 3D map use, schema above keeps position/color/display priority central. Useful future additions:

- `pm_ra_mas_yr`, `pm_dec_mas_yr`, `radial_velocity_km_s`: animate stellar motion or calculate epoch shifts.
- `epoch`: coordinate epoch, often J2000 or Gaia epoch 2016.0.
- `apparent_mag_v` or `apparent_mag_g`: useful for label/display importance.
- `constellation`: useful for grouping/filter UI.
- systems: `separation_arcsec`, `period_yr`, `semi_major_axis_au` if orbit visualization matters.

## Helper function guide

Core writers:

- `getStar(id, rank=4, parent_uid=None)`: query SIMBAD, build one `single_star` JSON, write to `public/data/stars/`.
- `getStarSystem(id, [child, ...], rank=4, parent_uid=None)`: query system, allocate system `uid`, write child stars, then write system JSON with `child_uids`. First child may fall back to parent/system numeric data.

UID / files:

- `next_uid()`: returns stable numeric uid from notebook cache.
- `cache_keys()`: list cached SIMBAD query keys.
- `remove_cache_entry("id", ...)`: delete exact cached SIMBAD query entries.
- `remove_cache_matching("text")`: delete cached SIMBAD entries by substring.
- `clear_simbad_cache(confirm=True)`: clear SIMBAD query cache.
- `set_next_uid(uid)`: manually reset next uid if you deleted JSON files.
- `write_json_entry(entry, filename_base=...)`: writes JSON. Filename uses proper name → latinized Bayer → Flamsteed, but object id is numeric `uid`.
- `find_existing_star_by_simbad_data(data, ids)` / `reuse_existing_star_if_found(data, ids)`: scan existing generated JSON by catalog ids and SIMBAD aliases to avoid duplicate stars. Existing files are reused, never rewritten.
- `find_entry_by_uid(uid)`: locate generated JSON by numeric uid.
- `remove_entry_by_uid(uid, confirm=True)`: delete generated JSON by uid and unlink from systems; if deleting a system, children become standalone rank-4 stars.
- `unique_path(name)`, `slugify(text)`: filename helpers.

Names and constellation:

- `proper_name_from_ids(ids_blob)`: chooses longest available `NAME ...` value.
- `find_bayer_flamsteed(ids_blob, main_id)`: searches all SIMBAD ids together; component-letter Bayer (`alf Cen A`) wins over superscript/numeric Bayer (`alf01 Cen`).
- `bayer_abbr_to_greek(abbr)`: `alf Cen A` → `α Centauri A`; `alf01 Cen` → `α¹ Centauri`.
- `bayer_abbr_to_latin(abbr)`: filename-friendly Bayer, e.g. `alpha Centauri A`.
- `make_bayer_abbr(greek, con_abbr, suffix=None, component=None)`: build SIMBAD-style Bayer abbreviation.
- `flamsteed_abbr_to_name(abbr)`, `make_flamsteed_abbr(number, con_abbr)`: Flamsteed helpers.
- `constellation_index_from_ids(ids_blob, main_id)`: returns IAU constellation index by alphabetical abbreviation; `And == 0`, unknown `-1`.

Superscripts / text cleanup:

- `superscript_to_ascii(text)`: `α¹ Cen` → `α1 Cen`.
- `ascii_to_superscript(text)`: `12` → `¹²`.
- `strip_star_prefix(id)`: removes SIMBAD `* ` prefix.
- `normalize_designation(id)`: removes `* ` and normalizes superscripts.

Astro calculations:

- `parallax_to_distance_pc(parallax_mas)` and `distance_pc_to_parallax_mas(distance_pc)`.
- `apparent_to_absolute_mag(mag, distance_pc)`.
- `bv_to_teff_k(bv)`, `teff_to_rgb(teff_k)`, `bv_to_rgb(bv)`.
- `spectral_type_to_teff_k(sp)`: rough fallback color estimate.
- `infer_evolution_stage(spectral_type, otype)`: broad stage (`dwarf`, `giant`, `supergiant`, `remnant`).
- `infer_star_type(spectral_type, otype)`: specific display class (`red giant`, `white dwarf`, `red dwarf`).


In [6]:
getStar("Sol", rank=0)

print()

wrote uid=6 public/data/stars/sol_2.json



In [7]:
getStar("alf Cen C", rank=0)

print()

reused existing uid=2 public/data/stars/proxima_centauri.json



In [74]:
getStarSystem("alpha Centauri", ["alpha Cen A", "alpha Cen B", "alpha Cen C"], rank=0)


wrote uid=4 public/data/stars/rigel_kentaurus.json
wrote uid=5 public/data/stars/toliman.json
reused existing uid=2 public/data/stars/proxima_centauri.json
wrote uid=3 public/data/stars/alpha_centauri.json


{'object_type': 'star_system',
 'uid': 3,
 'constellation': 17,
 'names': {'proper_name': None,
  'bayer_abbr': 'alf Cen',
  'bayer_name': 'Alpha Centauri',
  'bayer_greek': 'α Centauri',
  'flamsteed_abbr': None,
  'flamsteed_name': None},
 'ids': {'simbad_main_id': '* alf Cen',
  'hip_id': None,
  'hd_id': None,
  'gaia_dr2_id': None,
  'gaia_dr3_id': None,
  'gj_id': None,
  'tyc_id': None,
  '2mass_id': None},
 'is_child': False,
 'child_uids': [4, 5, 2],
 'remarks': None,
 'rank': 0}

In [73]:
remove_entry_by_uid(5, confirm=True)

uid=5 not found


In [8]:
set_next_uid(6)

next_uid=6
